## Expected data files
The notebook searches these locations for the files:
- `../data/train.jsonl`
- `../data/train.tables.jsonl`
- `../data/test.jsonl`
- `../data/test.tables.jsonl`

## 1. Install dependencies

In [1]:
%pip install -q transformers datasets sentencepiece accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and reproducibility

In [2]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 3. Load the dataset
- If rise error check file path.

In [3]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

train_data = read_jsonl("../data/train.jsonl")
train_tables = read_jsonl("../data/train.tables.jsonl")
test_data = read_jsonl("../data/test.jsonl")
test_tables = read_jsonl("../data/test.tables.jsonl")

print(f"Training questions: {len(train_data):,}")
print(f"Training tables:    {len(train_tables):,}")
print(f"Test questions:     {len(test_data):,}")
print(f"Test tables:        {len(test_tables):,}")

Training questions: 56,355
Training tables:    18,585
Test questions:     15,878
Test tables:        5,230


## 4. Inspect one example

In [25]:
train_table_id = {table["id"]: table for table in train_tables}
test_table_id = {table["id"]: table for table in test_tables}

example = train_data[0]
example_table = train_table_id[example["table_id"]]

print("Question:", example["question"])
print("Table ID:", example["table_id"])
print("Headers:", example_table["header"])
print("Structured SQL:", example["sql"])

display(pd.DataFrame(
    example_table["rows"][:5],
    columns=example_table["header"]
))

Question: Tell me what the notes are for South Australia 
Table ID: 1-1000181-1
Headers: ['State/territory', 'Text/background colour', 'Format', 'Current slogan', 'Current series', 'Notes']
Structured SQL: {'sel': 5, 'conds': [[3, 0, 'SOUTH AUSTRALIA']], 'agg': 0}


,State/territory,Text/background colour,Format,Current slogan,Current series,Notes
0,Australian Capital Territory,blue/white,Yaa·nna,ACT · CELEBRATION OF A CENTURY 2013,YIL·00A,Slogan screenprinted on plate
1,New South Wales,black/yellow,aa·nn·aa,NEW SOUTH WALES,BX·99·HI,No slogan on current series
2,New South Wales,black/white,aaa·nna,NSW,CPX·12A,Optional white slimline series
3,Northern Territory,ochre/white,Ca·nn·aa,NT · OUTBACK AUSTRALIA,CB·06·ZZ,New series began in June 2011
4,Queensland,maroon/white,nnn·aaa,QUEENSLAND · SUNSHINE STATE,999·TLG,Slogan embossed on plate


## 5. Convert WikiSQL annotations to SQL

In [26]:
AGGREGATIONS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
OPERATORS = ["=", ">", "<"]

def sql_literal(value):
    # WikiSQL condition values are commonly strings. Escape apostrophes for SQL.
    if value is None:
        return "NULL"
    text = str(value).replace("'", "''")
    return f"'{text}'"

def make_sql(item, table):
    annotation = item["sql"]
    headers = table["header"]

    select_index = int(annotation["sel"])
    agg_index = int(annotation["agg"])

    if not 0 <= select_index < len(headers):
        raise ValueError(f"Invalid SELECT column index: {select_index}")
    if not 0 <= agg_index < len(AGGREGATIONS):
        raise ValueError(f"Invalid aggregation index: {agg_index}")

    select_column = headers[select_index]
    aggregation = AGGREGATIONS[agg_index]
    select_expression = (
        f"{aggregation}({select_column})" if aggregation else select_column
    )
    query = f"SELECT {select_expression}"

    conditions = []
    for condition in annotation.get("conds", []):
        column_index, operator_index, value = condition
        column_index = int(column_index)
        operator_index = int(operator_index)

        if not 0 <= column_index < len(headers):
            raise ValueError(f"Invalid condition column index: {column_index}")
        if not 0 <= operator_index < len(OPERATORS):
            raise ValueError(f"Invalid condition operator index: {operator_index}")

        column = headers[column_index]
        operator = OPERATORS[operator_index]
        conditions.append(f"{column} {operator} {sql_literal(value)}")

    if conditions:
        query += " WHERE " + " AND ".join(conditions)

    return query

In [27]:
# Verify the conversion on one example.
print("Question:", example["question"])
print("Gold SQL:", make_sql(example, example_table))

Question: Tell me what the notes are for South Australia 
Gold SQL: SELECT Notes WHERE Current slogan = 'SOUTH AUSTRALIA'


## 6. Build consistent train and test examples

In [ ]:
def build_examples(question_data, tables_by_id):
    rows = []
    missing_table_ids = []

    for item in question_data:
        table = tables_by_id.get(item["table_id"])
        if table is None:
            missing_table_ids.append(item["table_id"])
            continue

        schema = " | ".join(str(header) for header in table["header"])
        question = re.sub(r"\\s+", " ", str(item["question"])).strip()
        input_text = (
            "translate English to SQL: "
            f"Question: {question} "
            f"Schema: {schema}"
        )

        rows.append({
            "question": question,
            "schema": schema,
            "input_text": input_text,
            "sql": make_sql(item, table),
        })

    if missing_table_ids:
        print(f"Warning: skipped {len(missing_table_ids)} examples with missing tables.")
    return pd.DataFrame(rows)

train_df = build_examples(train_data, train_table_id)
test_df = build_examples(test_data, test_table_id)

if train_df.empty:
    raise ValueError("No training examples were created. Check the dataset files.")

print("Train examples:", len(train_df))
print("Test examples:", len(test_df))
display(train_df[["question", "schema", "sql"]].head())

Train examples: 56355
Test examples: 15878


,question,schema,sql
0,Tell me what the notes are for South Australia,State/territory | Text/background colour | For...,SELECT Notes WHERE Current slogan = 'SOUTH AUS...
1,What is the current series where the new serie...,State/territory | Text/background colour | For...,SELECT Current series WHERE Notes = 'New serie...
2,What is the format for South Australia?,State/territory | Text/background colour | For...,SELECT Format WHERE State/territory = 'South A...
3,Name the background colour for the Australian ...,State/territory | Text/background colour | For...,SELECT Text/background colour WHERE State/terr...
4,how many times is the fuel propulsion is cng?,Order Year | Manufacturer | Model | Fleet Seri...,SELECT COUNT(Fleet Series (Quantity)) WHERE Fu...


## 7. Check data quality

In [8]:
print("Missing values in training data:")
display(train_df.isna().sum().to_frame("missing_count"))

print("Duplicate question/schema/SQL rows:", train_df.duplicated(
    subset=["input_text", "sql"]
).sum())

assert train_df["input_text"].notna().all()
assert train_df["sql"].notna().all()
assert train_df["input_text"].str.len().gt(0).all()
assert train_df["sql"].str.len().gt(0).all()

Missing values in training data:


,missing_count
question,0
schema,0
input_text,0
sql,0


Duplicate question/schema/SQL rows: 168


## 8. Tokenize inputs and SQL targets

In [9]:
MODEL_NAME = "t5-small"
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    target_tokens = tokenizer(
        text_target=batch["sql"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = target_tokens["input_ids"]
    return model_inputs

# Split rows before tokenization to keep the evaluation split separate.
from sklearn.model_selection import train_test_split

train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    shuffle=True,
)

train_dataset = Dataset.from_pandas(
    train_part[["input_text", "sql"]].reset_index(drop=True),
    preserve_index=False,
).map(tokenize_batch, batched=True, remove_columns=["input_text", "sql"])

val_dataset = Dataset.from_pandas(
    val_part[["input_text", "sql"]].reset_index(drop=True),
    preserve_index=False,
).map(tokenize_batch, batched=True, remove_columns=["input_text", "sql"])

print("Training rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))
print(train_dataset[0].keys())

Map:   0%|          | 0/50719 [00:00<?, ? examples/s]

Map:   0%|          | 0/5636 [00:00<?, ? examples/s]

Training rows: 50719
Validation rows: 5636
dict_keys(['input_ids', 'attention_mask', 'labels'])


## 9. Load the model and configure training

In [17]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="./finassist_t5_wikisql",
    num_train_epochs=4,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## 10. Train the Transformer

In [18]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss
1,0.256808,0.171830
2,0.200666,0.143962
3,0.180200,0.133808
4,0.185474,0.131229


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=25360, training_loss=0.24277446333918284, metrics={'train_runtime': 5358.8623, 'train_samples_per_second': 37.858, 'train_steps_per_second': 4.732, 'total_flos': 3781173962735616.0, 'train_loss': 0.24277446333918284, 'epoch': 4.0})


## 11. Save the trained model

In [28]:
MODEL_OUTPUT_DIR = Path("./finassist_t5_wikisql/final_model")
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('finassist_t5_wikisql\\final_model\\tokenizer_config.json',
 'finassist_t5_wikisql\\final_model\\tokenizer.json')

## 12. Generate SQL for a single question

In [20]:
def generate_sql(question, schema, model=model, tokenizer=tokenizer):
    input_text = (
        "translate English to SQL: "
        f"Question: {question} "
        f"Schema: {schema}"
    )
    model_inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    device = next(model.parameters()).device
    model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

    model.eval()
    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(generated[0], skip_special_tokens=True).strip()

sample_question = test_df.iloc[0]["question"] if not test_df.empty else train_df.iloc[0]["question"]
sample_schema = test_df.iloc[0]["schema"] if not test_df.empty else train_df.iloc[0]["schema"]

print("Question:", sample_question)
print("Schema:", sample_schema)
print("Generated SQL:", generate_sql(sample_question, sample_schema))

Question: What is terrence ross' nationality
Schema: Player | No. | Nationality | Position | Years in Toronto | School/Club Team
Generated SQL: SELECT Nationality WHERE Player = 'terence ross'


## 13. Evaluate on the held-out test set

In [21]:
def normalize_sql(sql):
    return re.sub(r"\\s+", " ", str(sql).strip().lower())

def predict_dataframe(frame, batch_size=8):
    predictions = []
    model.eval()
    device = next(model.parameters()).device

    for start in range(0, len(frame), batch_size):
        texts = frame["input_text"].iloc[start:start + batch_size].tolist()
        batch = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        )
        batch = {key: value.to(device) for key, value in batch.items()}

        with torch.no_grad():
            generated = model.generate(
                **batch,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=4,
                early_stopping=True,
            )

        predictions.extend(
            tokenizer.batch_decode(generated, skip_special_tokens=True)
        )
    return predictions

if test_df.empty:
    print("No test examples available; check the test JSONL files.")
else:
    test_df["predicted_sql"] = predict_dataframe(test_df)
    test_df["gold_normalized"] = test_df["sql"].map(normalize_sql)
    test_df["pred_normalized"] = test_df["predicted_sql"].map(normalize_sql)

    exact_match_accuracy = (
        test_df["gold_normalized"] == test_df["pred_normalized"]
    ).mean()

    print(f"Test exact-match accuracy: {exact_match_accuracy:.4f}")
    display(test_df[["question", "sql", "predicted_sql"]].head(10))

Test exact-match accuracy: 0.5323


,question,sql,predicted_sql
0,What is terrence ross' nationality,SELECT Nationality WHERE Player = 'Terrence Ross',SELECT Nationality WHERE Player = 'terence ross'
1,What clu was in toronto 1995-96,SELECT School/Club Team WHERE Years in Toronto...,SELECT School/Club Team WHERE Years in Toronto...
2,which club was in toronto 2003-06,SELECT School/Club Team WHERE Years in Toronto...,SELECT School/Club Team WHERE Years in Toronto...
3,how many schools or teams had jalen rose,SELECT COUNT(School/Club Team) WHERE Player = ...,SELECT COUNT(School/Club Team) WHERE Player = ...
4,Where was Assen held?,SELECT Round WHERE Circuit = 'Assen',SELECT Circuit WHERE Race winner = 'Assen'
5,What was the number of race that Kevin Curtain...,SELECT COUNT(No) WHERE Pole Position = 'Kevin ...,SELECT No WHERE Race winner = 'Kevin Curtain'
6,What was the date of the race in Misano?,SELECT Date WHERE Circuit = 'Misano',SELECT Date WHERE Circuit ='misano'
7,How many different positions did Sherbrooke Fa...,SELECT COUNT(Position) WHERE College/junior/cl...,SELECT COUNT(Position) WHERE Player = 'Sherbro...
8,What are the nationalities of the player picke...,SELECT Nationality WHERE College/junior/club t...,SELECT Nationality WHERE NHL team = 'Thomas Ba...
9,How many different college/junior/club teams p...,SELECT COUNT(College/junior/club team) WHERE N...,SELECT COUNT(College/junior/club team) WHERE N...
